# Dense duck formation from the original frozen PINN

42 particles from six fixed wall sites; 64 particles from independent wall positions. Original force law, diameter, flow network and acceptance thresholds are retained. Filled silhouettes reconstruct simulated boundary landmarks; physical disks are shown separately. No neural training.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, json, time, math, csv, shutil, itertools, html

# Kaggle/Colab: enable Internet for the first dependency installation, and GPU.
required = {'numpy': 'numpy', 'scipy': 'scipy', 'matplotlib': 'matplotlib',
            'torch': 'torch', 'IPython': 'ipython'}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import least_squares, minimize_scalar
from scipy.stats import qmc
from IPython.display import display, HTML, Image, FileLink

BASE = (Path('/kaggle/working') if Path('/kaggle/working').exists() else
        Path('/content') if Path('/content').exists() else Path.cwd())
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

# All quantities are nondimensional: x/L0, velocity/U0, t/(L0/U0).
# Re = rho_f U0 L0 / mu; U0 is the centreline speed, NOT the mean speed.
L, H, U_C = 2.2, 0.30, 1.0
RE = 10.0
DIAMETER = 0.025
DENSITY_RATIO = 1000.0   # rho_particle/rho_fluid; retained rigid-particle model.
DRAG_LAW = 'schiller-naumann'  # or 'stokes'
LAUNCH_GAP = 1e-5       # centre starts radius+gap inside the appropriate wall.

CONTROL_NAMES = ('x_bottom', 'u_bottom', 'v_bottom',
                 'x_top', 'u_top', 'v_top')
INPUT_NAMES = ('x', 'y', *CONTROL_NAMES)  # EXACTLY eight MLP inputs.
CONTROL_BOUNDS = np.array([
    [0.0, L], [-0.40, 1.50], [0.15, 1.20],
    [0.0, L], [-0.40, 1.50], [-1.20, -0.15],
], dtype=float)
# Both wall launch positions span the FULL interval [0,L], including endpoints.
# v_bottom > 0 and v_top < 0 inject INTO the channel. Speeds are Cartesian.

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float32
WIDTH, DEPTH = 64, 4
ADAM_STEPS, SAMPLES = 3000, 768
LEARNING_RATE, CHECK_EVERY = 7e-4, 200
LBFGS_ROUNDS, LBFGS_INNER, LBFGS_SAMPLES = 6, 25, 2048
HOURS = 6.0  # Adam time cap; L-BFGS and validation follow. None disables cap.
RESUME_CHECKPOINT = None  # Only a checkpoint from THIS rectangular formulation.

TRAJECTORY_TMAX = 20.0
TRAJECTORY_MAX_STEP = 0.035
ODE_RTOL, ODE_ATOL = 2e-7, 2e-9
VELOCITY_REL_TOL, PRESSURE_REL_TOL = 2e-3, 5e-3
TRANSVERSE_ABS_TOL, CONDITIONING_ABS_TOL = 2e-3, 2e-3
ALLOW_UNVALIDATED_OPTIMIZATION = False

assert min(L, H, U_C, RE, DIAMETER, DENSITY_RATIO, LAUNCH_GAP) > 0
assert DIAMETER + 2*LAUNCH_GAP < H
assert CONTROL_BOUNDS.shape == (6, 2)
assert np.isfinite(CONTROL_BOUNDS).all() and np.all(np.diff(CONTROL_BOUNDS, axis=1) > 0)
assert np.array_equal(CONTROL_BOUNDS[[0, 3]], [[0, L], [0, L]])
assert CONTROL_BOUNDS[2, 0] > 0 and CONTROL_BOUNDS[5, 1] < 0
assert DRAG_LAW in ('stokes', 'schiller-naumann')
assert min(WIDTH, DEPTH, SAMPLES, ADAM_STEPS, CHECK_EVERY, LBFGS_INNER) >= 1
assert LBFGS_ROUNDS >= 0 and LBFGS_SAMPLES >= 1
assert HOURS is None or (np.isfinite(HOURS) and HOURS > 0)
assert min(TRAJECTORY_TMAX, TRAJECTORY_MAX_STEP, ODE_RTOL, ODE_ATOL) > 0

torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)
rng = np.random.default_rng(SEED)
CONTROL_MID = CONTROL_BOUNDS.mean(axis=1)
CONTROL_SPAN = np.ptp(CONTROL_BOUNDS, axis=1)
CONTROL_CORNERS = np.array(list(itertools.product([0., 1.], repeat=6)))
LOSS_WEIGHTS = dict(continuity=20., momentum=1., developed=5., invariance=25., flux=5.)
PRESSURE_SCALE = 8*U_C*L/(RE*H**2)
SCOPE = 'rectangle-two-wall-launchers-eight-inputs-v1'
CONFIG = dict(scope=SCOPE, inputs=list(INPUT_NAMES), controls=list(CONTROL_NAMES),
    bounds=CONTROL_BOUNDS.tolist(), L=L, H=H, U_C=U_C, re=RE,
    diameter=DIAMETER, density_ratio=DENSITY_RATIO, drag=DRAG_LAW,
    launch_gap=LAUNCH_GAP, width=WIDTH, depth=DEPTH,
    coupling='one-way; launch controls do not alter the carrier flow', seed=SEED,
    dtype=str(DTYPE), adam_steps=ADAM_STEPS, samples=SAMPLES,
    lbfgs_rounds=LBFGS_ROUNDS, lbfgs_inner=LBFGS_INNER, loss_weights=LOSS_WEIGHTS,
    time_scale='L0/U0; Re is based on L0 and centreline U0')
RUN = BASE / ('rectangular_wall_launchers_' + str(time.time_ns()))
RUN.mkdir(parents=True)
(RUN/'config.json').write_text(json.dumps(CONFIG, indent=2))
print('Run:', RUN)
print('MLP:', list(INPUT_NAMES), '-> [u,v,p]')
print('Fixed Re/d/rho ratio:', RE, DIAMETER, DENSITY_RATIO)
print('Stokes relaxation time:', DENSITY_RATIO*RE*DIAMETER**2/18)

def exact_flow(xy, controls=None):
    """Independent plane Poiseuille reference; not an interior training dataset."""
    xy = np.atleast_2d(np.asarray(xy, float))
    eta = xy[:, 1]/H
    return np.column_stack((4*U_C*eta*(1-eta), np.zeros(len(xy)),
                            8*U_C*(L-xy[:, 0])/(RE*H**2)))


class RectangularFlow(nn.Module):
    """Eight physical inputs. Hard velocity BCs on all four sides; p(L,y)=0."""
    def __init__(self, width=WIDTH, depth=DEPTH):
        super().__init__()
        self.register_buffer('lower', torch.tensor([0, 0, *CONTROL_BOUNDS[:, 0]], dtype=DTYPE))
        self.register_buffer('span', torch.tensor([L, H, *CONTROL_SPAN], dtype=DTYPE))
        layers = [nn.Linear(8, width), nn.Tanh()]
        for _ in range(depth-1): layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 3)]
        self.net = nn.Sequential(*layers)
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)
        # Nonzero random interior correction; exact pressure is not initialized.
        with torch.no_grad(): self.net[-1].weight.mul_(0.15)

    def forward(self, z):
        raw = self.net(2*(z-self.lower)/self.span-1)
        X, Y = z[:, 0]/L, z[:, 1]/H
        parabola = 4*U_C*Y*(1-Y)
        cap = X*(1-X)
        return torch.stack((parabola*(1+cap*raw[:, 0]),
                            parabola*cap*raw[:, 1],
                            PRESSURE_SCALE*(1-X)*raw[:, 2]), dim=1)


def flow_inputs(xy, controls):
    xy = np.atleast_2d(np.asarray(xy, float))
    c = np.asarray(controls, float)
    if c.ndim == 1: c = np.broadcast_to(c, (len(xy), 6))
    if xy.shape[1] != 2 or c.shape != (len(xy), 6):
        raise ValueError('Expected Nx2 coordinates and six shared or Nx6 controls.')
    return np.column_stack((xy, c))


def sample_inputs(count, generator, device=DEVICE):
    unit = generator.random((count, 8))
    # Train the whole control box, including ALL 64 corners, every full batch.
    # Duplicate anchors use independent spatial coordinates, not duplicate PDE points.
    if count >= 64:
        unit[:64, 2:] = CONTROL_CORNERS
        if count >= 128: unit[64:128, 2:] = CONTROL_CORNERS
    elif count >= 4:
        unit[:4, 2] = [0, 0, 1, 1]
        unit[:4, 5] = [0, 1, 0, 1]
    lo = np.array([0, 0, *CONTROL_BOUNDS[:, 0]])
    span = np.array([L, H, *CONTROL_SPAN])
    return torch.tensor(lo+span*unit, dtype=DTYPE, device=device)


def derivative(a, z):
    return torch.autograd.grad(a.sum(), z, create_graph=True)[0]


def flow_terms(net, batch):
    z = batch.detach().clone().requires_grad_(True)
    value = net(z)
    u, v, p = value.unbind(1)
    gu, gv, gp = (derivative(a, z) for a in (u, v, p))
    uxx, uyy = derivative(gu[:, 0], z)[:, 0], derivative(gu[:, 1], z)[:, 1]
    vxx, vyy = derivative(gv[:, 0], z)[:, 0], derivative(gv[:, 1], z)[:, 1]
    # Steady nondimensional incompressible NS. Spatial derivatives only x,y.
    continuity = (H*(gu[:, 0]+gv[:, 1])/U_C).square().mean()
    rx = H**2*(RE*(u*gu[:, 0]+v*gu[:, 1]+gp[:, 0])-uxx-uyy)/U_C
    ry = H**2*(RE*(u*gv[:, 0]+v*gv[:, 1]+gp[:, 1])-vxx-vyy)/U_C
    momentum = (rx.square()+ry.square()).mean()
    # Fully developed flow: du/dx=dv/dx=0, v=0.
    developed = ((L*gu[:, 0]/U_C).square().mean() +
                 (L*gv[:, 0]/U_C).square().mean() + (v/U_C).square().mean())
    # In one-way coupling, changing launcher controls must leave fluid unchanged.
    half_span = z.new_tensor(CONTROL_SPAN/2)
    invariance = sum((g[:, 2:]*half_span/scale).square().mean()
                     for g, scale in [(gu, U_C), (gv, U_C), (gp, PRESSURE_SCALE)])
    # Gaussian flux, using independent control tuples from this batch.
    nodes, weights = np.polynomial.legendre.leggauss(12)
    count = min(4, len(z))
    controls = z[:count, 2:].detach().repeat_interleave(36, dim=0)
    xs = z.new_tensor([.15, .5, .85]).repeat_interleave(12).repeat(count)*L
    ys = z.new_tensor((nodes+1)*H/2).repeat(count*3)
    zz = torch.cat((torch.stack((xs, ys), 1), controls), dim=1)
    quadrature = (net(zz)[:, 0].reshape(count*3, 12)*z.new_tensor(weights)).sum(1)*H/2
    flux = (quadrature/(2*U_C*H/3)-1).square().mean()
    total = (LOSS_WEIGHTS['continuity']*continuity+LOSS_WEIGHTS['momentum']*momentum+
             LOSS_WEIGHTS['developed']*developed+LOSS_WEIGHTS['invariance']*invariance+
             LOSS_WEIGHTS['flux']*flux)
    return dict(total=total, continuity=continuity, momentum=momentum,
                developed=developed, invariance=invariance, flux=flux)


def evaluate_flow(net, xy, controls=CONTROL_MID, batch_size=8192):
    packed = flow_inputs(xy, controls)
    parameter = next(net.parameters())
    chunks = []
    with torch.no_grad():
        for start in range(0, len(packed), batch_size):
            z = torch.as_tensor(packed[start:start+batch_size],
                                dtype=parameter.dtype, device=parameter.device)
            chunks.append(net(z).cpu().numpy())
    return np.vstack(chunks)


def load_flow(path, device='cpu'):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    saved = checkpoint['config']
    for key in ['scope', 'inputs', 'bounds', 'L', 'H', 'U_C', 're', 'diameter',
                'density_ratio', 'drag', 'launch_gap', 'width', 'depth', 'dtype']:
        if saved.get(key) != CONFIG.get(key):
            raise ValueError(f'Checkpoint mismatch: {key}. Run matching settings first.')
    net = RectangularFlow(saved['width'], saved['depth']).to(device=device, dtype=DTYPE)
    net.load_state_dict(checkpoint['model'])
    return net.eval(), checkpoint

def validate_model(net, nx=81, ny=33, random_controls=16, save=True):
    x = np.linspace(0, L, nx)
    y = np.linspace(0, H, ny)
    X, Y = np.meshgrid(x, y)
    xy = np.column_stack((X.ravel(), Y.ravel()))
    ref = exact_flow(xy)
    corners = np.array(list(itertools.product([0.0, 1.0], repeat=6)))
    bank = np.vstack((CONTROL_MID, CONTROL_BOUNDS[:, 0] + corners * CONTROL_SPAN, CONTROL_BOUNDS[:, 0] + np.random.default_rng(SEED + 82).random((random_controls, 6)) * CONTROL_SPAN))
    baseline = evaluate_flow(net, xy, CONTROL_MID)
    rows = []
    for i, controls in enumerate(bank):
        pred = evaluate_flow(net, xy, controls)
        rows.append(dict(case=i, controls=controls.tolist(), velocity_relative_l2=float(np.linalg.norm(pred[:, :2] - ref[:, :2]) / np.linalg.norm(ref[:, :2])), pressure_relative_l2=float(np.linalg.norm(pred[:, 2] - ref[:, 2]) / np.linalg.norm(ref[:, 2])), transverse_max_abs=float(np.max(np.abs(pred[:, 1]))), max_condition_velocity_delta=float(np.max(np.abs(pred[:, :2] - baseline[:, :2]))), max_condition_pressure_delta_normalized=float(np.max(np.abs(pred[:, 2] - baseline[:, 2])) / PRESSURE_SCALE)))
    worst = {k: max((row[k] for row in rows)) for k in rows[0] if k not in ['case', 'controls']}
    boundary_xy = np.vstack((np.column_stack((x, np.zeros(nx))), np.column_stack((x, np.full(nx, H))), np.column_stack((np.zeros(ny), y)), np.column_stack((np.full(ny, L), y))))
    b = evaluate_flow(net, boundary_xy)
    boundary_error = float(np.max(np.abs(b[:, :2] - exact_flow(boundary_xy)[:, :2])))
    nodes, weights = np.polynomial.legendre.leggauss(32)
    flux_errors = []
    for sx in np.linspace(0, L, 11):
        uv = evaluate_flow(net, np.column_stack((np.full(32, sx), (nodes + 1) * H / 2)))
        flux_errors.append(abs(np.dot(uv[:, 0], weights) * H / 2 / (2 * U_C * H / 3) - 1))
    passed = worst['velocity_relative_l2'] <= VELOCITY_REL_TOL and worst['pressure_relative_l2'] <= PRESSURE_REL_TOL and (worst['transverse_max_abs'] <= TRANSVERSE_ABS_TOL) and (worst['max_condition_velocity_delta'] <= CONDITIONING_ABS_TOL) and (worst['max_condition_pressure_delta_normalized'] <= CONDITIONING_ABS_TOL) and (max(flux_errors) <= VELOCITY_REL_TOL) and (boundary_error <= 1e-05)
    report = dict(passed=bool(passed), cases=len(bank), grid=[nx, ny], worst=worst, velocity_boundary_max_error=boundary_error, flux_max_relative_error=max(flux_errors), thresholds=dict(velocity_relative_l2=VELOCITY_REL_TOL, pressure_relative_l2=PRESSURE_REL_TOL, transverse_abs=TRANSVERSE_ABS_TOL, conditioning_abs=CONDITIONING_ABS_TOL), reference='Analytical plane Poiseuille solution; no interior reference samples in training', cases_include='All 64 corners of the six-dimensional control box, its midpoint, and random controls', scope='Finite validation samples, not a proof over the continuum', details=rows)
    if save:
        (RUN / 'validation.json').write_text(json.dumps(report, indent=2))
        np.savez_compressed(RUN / 'flow_validation.npz', xy=xy, predicted=baseline, reference=ref, control_bank=bank)
    return report
def check_controls(controls):
    c = np.asarray(controls, float)
    if c.shape != (6,) or not np.isfinite(c).all():
        raise ValueError('Each trajectory needs six finite flow-conditioning slots; active launch triplet is particle-specific.')
    if np.any(c < CONTROL_BOUNDS[:, 0] - 1e-12) or np.any(c > CONTROL_BOUNDS[:, 1] + 1e-12):
        raise ValueError('Launcher controls are outside the trained box.')
    return np.clip(c, CONTROL_BOUNDS[:, 0], CONTROL_BOUNDS[:, 1])

def particle_acceleration(fluid, particle_velocity):
    slip = np.asarray(fluid) - np.asarray(particle_velocity)
    rep = RE * DIAMETER * np.linalg.norm(slip)
    factor = 1.0
    if DRAG_LAW == 'schiller-naumann':
        factor = 1 + 0.15 * rep ** 0.687 if rep < 1000 else 0.44 * rep / 24
    return 18 * factor * slip / (DENSITY_RATIO * RE * DIAMETER ** 2)

class ParticlePath:

    def __init__(self, solution, side, controls, status):
        self.sol, self.side = (solution, side)
        self.controls, self.status = (controls.copy(), status)
        self.end = float(solution.t[-1])
        self.ok = bool(solution.success and np.isfinite(solution.y).all())

    def state(self, ages, strict=True):
        a = np.atleast_1d(np.asarray(ages, float))
        if strict and (np.any(a < -1e-09) or np.any(a > self.end + 1e-08)):
            raise ValueError('Requested a particle state outside its physical trajectory.')
        return self.sol.sol(np.clip(a, 0, self.end)).T

def trace_particle(field, controls, side, tmax=TRAJECTORY_TMAX, rtol=ODE_RTOL, atol=ODE_ATOL, max_step=TRAJECTORY_MAX_STEP):
    """field(xy, particle_controls) -> Nx2 or Nx3; no particle feedback to fluid."""
    c = check_controls(controls)
    if side not in (0, 1):
        raise ValueError('side=0 is bottom, side=1 is top.')
    j = 0 if side == 0 else 3
    y0 = DIAMETER / 2 + LAUNCH_GAP if side == 0 else H - DIAMETER / 2 - LAUNCH_GAP
    initial = [c[j], y0, c[j + 1], c[j + 2]]
    calls = 0

    def rhs(t, state):
        nonlocal calls
        calls += 1
        if calls > 100000:
            raise RuntimeError('ODE RHS budget exhausted.')
        xy = np.clip(state[:2], [0, 0], [L, H])[None, :]
        fluid = np.asarray(field(xy, c))[0, :2]
        if not np.isfinite(fluid).all():
            raise FloatingPointError('Non-finite fluid velocity.')
        acc = particle_acceleration(fluid, state[2:])
        return [state[2], state[3], acc[0], acc[1]]

    def lower(t, z):
        return z[1] - DIAMETER / 2

    def upper(t, z):
        return H - DIAMETER / 2 - z[1]

    def inlet(t, z):
        return z[0]

    def outlet(t, z):
        return L - z[0]
    events = [lower, upper, inlet, outlet]
    for event in events:
        event.terminal, event.direction = (True, -1)
    tau = DENSITY_RATIO * RE * DIAMETER ** 2 / 18
    method = 'Radau' if tau < max_step / 3 else 'RK45'
    solution = solve_ivp(rhs, (0, tmax), initial, events=events, dense_output=True, method=method, rtol=rtol, atol=atol, max_step=max_step)
    hit = next((i for i, values in enumerate(solution.t_events) if len(values)), None)
    status = ['lower_wall', 'upper_wall', 'inlet_exit', 'outlet'][hit] if hit is not None else 'time_limit' if solution.success else 'solver_failed'
    return ParticlePath(solution, side, c, status)

def direct_field(xy, controls):
    return evaluate_flow(model, xy, controls)[:, :2]

class FlowLookup:
    """2D speed table at midpoint controls; final verification uses all 8 inputs.

    This reduction is valid for the specified one-way fully developed flow only.
    Conditional drift is measured before optimization; the table is not a new MLP.
    """

    def __init__(self, net, nx=241, ny=121):
        self.x, self.y = (np.linspace(0, L, nx), np.linspace(0, H, ny))
        X, Y = np.meshgrid(self.x, self.y, indexing='ij')
        xy = np.column_stack((X.ravel(), Y.ravel()))
        values = evaluate_flow(net, xy, CONTROL_MID)[:, :2].reshape(nx, ny, 2)
        self.interpolate = RegularGridInterpolator((self.x, self.y), values, bounds_error=True)
        self.values = values

    def __call__(self, xy, controls=None):
        return self.interpolate(np.clip(np.atleast_2d(xy), [0, 0], [L, H]))

import base64
_checkpoint = RUN/'original_frozen_checkpoint.pt'
_checkpoint.write_bytes(base64.b64decode('...'))
model, _ = load_flow(_checkpoint, 'cpu')
model.eval().requires_grad_(False)
VALIDATION=validate_model(model)
assert VALIDATION['passed'], 'Original flow validation must pass.'
print('Original frozen checkpoint loaded; unchanged physical validation passed. No neural training.')
# Each row of the design vector is [x_launch, u_launch, v_launch, age_at_target].
# The wall side is selected separately from candidate trajectories for EACH particle.
SHAPES = ['triangle', 'square', 'circle_10', 's_8', 'chevron_7']
TARGET_CENTER = np.array([0.86*L, H/2])
TARGET_RADIUS = 0.075
S_WIDTH, S_HEIGHT = 0.13, 0.19
POSITION_TOLERANCE = 0.003
COLLISION_MARGIN = 0.001
LEAD_TIME = 0.25
AGE_MAX = min(5.0, TRAJECTORY_TMAX)
SEED_U_SPEEDS = (.25, .70, 1.20)
SEED_V_SPEEDS = np.linspace(.15, 1.20, 15)
CANDIDATES_PER_SIDE = 8
OPT_STARTS = 3
COARSE_MAX_NFEV, DIRECT_MAX_NFEV = 50, 35
COLLISION_SAMPLES = 61
PAIR_VERIFY_SAMPLES = 601
CLEARANCE_RESERVE = 0.0003  # optimize with slack; retain the original PASS threshold.
REFERENCE_TOLERANCE = 0.004
RUN_OPTIMIZATION = True

assert AGE_MAX > .02 and min(OPT_STARTS, COARSE_MAX_NFEV, DIRECT_MAX_NFEV) >= 1
assert COLLISION_SAMPLES >= 3 and PAIR_VERIFY_SAMPLES >= 21
assert min(POSITION_TOLERANCE, COLLISION_MARGIN, LEAD_TIME, REFERENCE_TOLERANCE) > 0
OPT_CONFIG = dict(shapes=SHAPES, center=TARGET_CENTER.tolist(), radius=TARGET_RADIUS,
    s_width=S_WIDTH, s_height=S_HEIGHT, position_tolerance=POSITION_TOLERANCE,
    collision_margin=COLLISION_MARGIN, clearance_reserve=CLEARANCE_RESERVE,
    lead_time=LEAD_TIME, age_max=AGE_MAX, starts=OPT_STARTS,
    seed_u_speeds=list(SEED_U_SPEEDS), seed_v_speeds=SEED_V_SPEEDS.tolist(),
    candidates_per_side=CANDIDATES_PER_SIDE, coarse_max_nfev=COARSE_MAX_NFEV,
    direct_max_nfev=DIRECT_MAX_NFEV, collision_samples=COLLISION_SAMPLES,
    reference_tolerance=REFERENCE_TOLERANCE,
    control_mode='independent x_launch, u_launch, v_launch and age for EVERY particle',
    side_mode='each particle selects a wall from its candidate bank; held fixed in each local solve',
    emission_time='T=max(age)+lead_time; release_i=T-age_i',
    flow_conditioning='active wall triplet from particle i; inactive triplet at CONTROL_MID',
    hardware='spatially programmable virtual wall ports, not two stationary nozzles')
(RUN/'optimization_config.json').write_text(json.dumps(OPT_CONFIG, indent=2))

from collections import OrderedDict
from scipy.optimize import brentq
class IndependentLauncherPlanner:
    """One launch triplet and one physical flight age per particle, never shared.

    The six conditioning slots of the existing flow PINN remain unchanged.
    For each trajectory the active wall's three slots carry THAT particle's
    launch settings and the unused wall's three slots remain at their midpoints.
    This is valid for the declared one-way, control-independent carrier flow.
    """

    def __init__(self, field, label, rtol=2e-06, atol=2e-08, max_step=0.06):
        self.field, self.label = (field, label)
        self.rtol, self.atol, self.max_step = (rtol, atol, max_step)
        self.cache, self.calls = (OrderedDict(), 0)

    def pack(self, controls, side, age):
        j = 3 * int(side)
        return np.r_[(np.asarray(controls)[j:j + 3] - CONTROL_BOUNDS[j:j + 3, 0]) / CONTROL_SPAN[j:j + 3], age / AGE_MAX]

    def physical(self, row, side):
        row = np.clip(np.asarray(row, float), 0, 1)
        j = 3 * int(side)
        controls = CONTROL_MID.copy()
        controls[j:j + 3] = CONTROL_BOUNDS[j:j + 3, 0] + row[:3] * CONTROL_SPAN[j:j + 3]
        return (controls, float(row[3] * AGE_MAX))

    def one_path(self, row, side):
        key = (int(side), *np.round(np.asarray(row), 12))
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        controls, age = self.physical(row, side)
        path = trace_particle(self.field, controls, int(side), tmax=max(age, 1e-08), rtol=self.rtol, atol=self.atol, max_step=self.max_step)
        if not path.ok:
            raise RuntimeError('Particle trajectory integration failed.')
        self.cache[key] = path
        self.calls += 1
        if len(self.cache) > 1024:
            self.cache.popitem(last=False)
        return path

    def endpoints(self, z, assignment):
        rows = np.asarray(z).reshape(len(assignment), 4)
        paths = [self.one_path(row, side) for row, side in zip(rows, assignment)]
        ages = rows[:, 3] * AGE_MAX
        actual = np.array([p.state([min(a, p.end)])[0, :2] for p, a in zip(paths, ages)])
        return (paths, ages, actual)

    def residual(self, z, assignment, targets):
        paths, ages, actual = self.endpoints(z, assignment)
        residual = list(((actual - targets) / H).ravel())
        residual.extend((3 * max(0.0, a - p.end) / AGE_MAX for p, a in zip(paths, ages)))
        required = DIAMETER + COLLISION_MARGIN + CLEARANCE_RESERVE
        for i in range(len(paths)):
            for j in range(i):
                backwards = np.linspace(0, min(ages[i], ages[j]), COLLISION_SAMPLES)
                pi = paths[i].state(ages[i] - backwards, strict=False)[:, :2]
                pj = paths[j].state(ages[j] - backwards, strict=False)[:, :2]
                distance = np.linalg.norm(pi - pj, axis=1)
                residual.extend(4 * np.maximum(0, required - distance) / H)
        return np.asarray(residual)

    def refine(self, z, assignment, targets, max_nfev):
        z = np.clip(np.asarray(z, float).ravel(), 0, 1)
        history, replay = ([], [])

        def record(point):
            paths, ages, _ = self.endpoints(point, assignment)
            controls = [self.physical(row, side)[0] for row, side in zip(np.asarray(point).reshape(-1, 4), assignment)]
            replay.append(independent_optimization_snapshot(paths, controls, assignment, ages, targets, self.label, len(replay)))

        def objective(point):
            residual = self.residual(point, assignment, targets)
            _, _, actual = self.endpoints(point, assignment)
            history.append(dict(evaluation=len(history), cost=float(np.dot(residual, residual) / 2), max_error=float(np.linalg.norm(actual - targets, axis=1).max()), actual=actual.tolist()))
            return residual

        def jacobian(point):
            base = self.residual(point, assignment, targets)
            record(point)
            jac = np.empty((len(base), len(point)))
            for k in range(len(point)):
                delta = 0.0002 if point[k] + 0.0002 <= 1 else -0.0002
                trial = point.copy()
                trial[k] += delta
                jac[:, k] = (self.residual(trial, assignment, targets) - base) / delta
            return jac
        result = least_squares(objective, z, jac=jacobian, bounds=(np.zeros(len(z)), np.ones(len(z))), max_nfev=max_nfev, ftol=5e-09, xtol=5e-08, gtol=5e-08)
        record(result.x)
        return (result.x, dict(message=result.message, nfev=int(result.nfev), cost=float(result.cost), optimizer_terminated=bool(result.success), history=history, continuous_decisions=int(len(z)), per_particle_decisions=4, replay=replay))

def independent_seed_bank(planner):
    """Reusable seed trajectories; a shape can draw each particle from ANY row.

    Because the physical developed flow is independent of x, a seed trajectory
    can first be translated in x. It is then integrated again at its actual
    position with the lookup, and later with all eight direct PINN inputs.
    Translation is an initializer only; it is never the accepted trajectory.
    """
    bank = []
    for side, ux, speed in itertools.product((0, 1), SEED_U_SPEEDS, SEED_V_SPEEDS):
        j = 3 * side
        c = CONTROL_MID.copy()
        c[j:j + 3] = [min(0.01, L / 100), ux, speed if side == 0 else -speed]
        if np.any(c < CONTROL_BOUNDS[:, 0]) or np.any(c > CONTROL_BOUNDS[:, 1]):
            continue
        path = trace_particle(planner.field, c, side, tmax=AGE_MAX, rtol=planner.rtol, atol=planner.atol, max_step=planner.max_step)
        if path.ok and path.end > 0.02:
            ages = np.linspace(0, path.end, 401)
            bank.append(dict(controls=c, side=side, path=path, ages=ages, points=path.state(ages)[:, :2]))
    if not bank:
        raise RuntimeError('No usable independent launch seeds.')
    print(f'Independent seed bank: {len(bank)} reusable paths; controls can differ for every particle.', flush=True)
    return bank

def options_for_particle(planner, bank, target):
    options = []
    for seed in bank:
        path, side, c = (seed['path'], seed['side'], seed['controls'].copy())
        ydiff = seed['points'][:, 1] - target[1]
        crossings = np.flatnonzero(ydiff[:-1] * ydiff[1:] <= 0)
        if not len(crossings):
            continue
        k = int(crossings[0])
        age = brentq(lambda a: float(path.state([a])[0, 1] - target[1]), seed['ages'][k], seed['ages'][k + 1], xtol=1e-11)
        if age < 0.02:
            continue
        state = path.state([age])[0]
        j = 3 * side
        c[j] += target[0] - state[0]
        if not 0 <= c[j] <= L:
            continue
        row = planner.pack(c, side, age)
        translated = planner.one_path(row, side)
        if translated.end < age - 1e-08:
            continue
        actual = translated.state([age])[0, :2]
        options.append(dict(row=row, side=side, path=translated, age=age, error=float(np.linalg.norm(actual - target))))
    if not options:
        raise RuntimeError(f'No seed reaches target {target.tolist()} within current bounds. Expand the seed grid or flight-age budget; this is not an impossibility proof.')
    chosen = []
    for side in (0, 1):
        group = sorted([o for o in options if o['side'] == side], key=lambda o: o['error'])
        if not group:
            continue
        selected = [group.pop(0)]
        while group and len(selected) < CANDIDATES_PER_SIDE:
            k = max(range(len(group)), key=lambda k: min((np.linalg.norm(group[k]['row'] - s['row']) for s in selected)) - 20 * group[k]['error'] / H)
            selected.append(group.pop(k))
        chosen.extend(selected)
    return chosen

def minimum_separation(paths, ages, samples=PAIR_VERIFY_SAMPLES, refine=True):
    """Distance only while both particles exist, using the SAME global clock."""
    minimum = float('inf')
    for i in range(len(paths)):
        for j in range(i):
            end = min(ages[i], ages[j])

            def distance(backwards):
                backwards = np.atleast_1d(backwards)
                a = paths[i].state(ages[i] - backwards)[:, :2]
                b = paths[j].state(ages[j] - backwards)[:, :2]
                return np.linalg.norm(a - b, axis=1)
            times = np.linspace(0, end, samples)
            d = distance(times)
            minimum = min(minimum, float(d.min()))
            if refine and end > 0:
                local = np.flatnonzero((d[1:-1] <= d[:-2]) & (d[1:-1] <= d[2:])) + 1
                for k in local:
                    result = minimize_scalar(lambda t: float(distance(t)[0]), bounds=(times[k - 1], times[k + 1]), method='bounded', options={'xatol': 1e-10})
                    minimum = min(minimum, float(result.fun))
    return minimum
def write_schedule(path, rows):
    if not rows:
        return
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
TARGET_CENTER=np.array([1.83,.14])


In [ ]:
import base64, zlib
_DENSE_PRECOMPUTED=json.loads(zlib.decompress(base64.b64decode('...')))
def sparse_trajectory_jacobian(planner,z,ports,targets,fixed_stations):
    """Sparse derivative of the unchanged shared-clock residual.

    Launch derivatives re-integrate just the affected particle. Flight-age
    derivatives use the original ODE velocities and the exact common-clock rule.
    No force term, acceptance threshold or particle geometry is altered.
    """
    from scipy.sparse import coo_matrix
    n=len(ports);dim=3 if fixed_stations else 4;S=COLLISION_SAMPLES
    rows=np.asarray(z).reshape(n,dim)
    paths,ages,actual=planner.endpoints(z,ports)
    f=np.linspace(0,1,S)
    backwards=np.minimum(ages[:,None],ages[None,:])[:,:,None]*f
    times=ages[:,None,None]-backwards
    states=np.array([p.state(times[i].ravel(),strict=False).reshape(n,S,4) for i,p in enumerate(paths)])
    positions=states[...,:2];velocities=states[...,2:]
    # The solver's strict=False display clamp is also respected at invalid iterates.
    velocities=velocities*(times<=np.array([p.end for p in paths])[:,None,None]+1e-9)[...,None]
    dif=positions-positions.swapaxes(0,1)
    distances=np.linalg.norm(dif,axis=-1)
    unit=dif/np.maximum(distances[...,None],1e-12)
    required=DIAMETER+COLLISION_MARGIN+CLEARANCE_RESERVE
    active=distances<required
    for i in range(n):active[i,i]=False
    ri=[];ci=[];vv=[]
    def add(row,col,values):
        row=np.atleast_1d(row);values=np.atleast_1d(values)
        mask=np.abs(values)>1e-15
        if np.any(mask):
            ri.extend(row[mask].tolist());ci.extend([col]*int(np.count_nonzero(mask)));vv.extend(values[mask].tolist())
    for i in range(n):
        j,k=np.nonzero(active[i])
        pair_rows=3*n+(np.maximum(i,j)*(np.maximum(i,j)-1)//2+np.minimum(i,j))*S+k
        pair_times=times[i,j,k]
        for control in range(dim-1):
            step=2e-4 if 'direct' in planner.label.lower() else 2e-5
            delta=step if rows[i,control]+step<=1 else -step
            q=rows[i].copy();q[control]+=delta
            perturbed=planner.one_path(q,ports[i])
            endpoint=perturbed.state([ages[i]],strict=False)[0,:2]
            add(np.array([2*i,2*i+1]),i*dim+control,(endpoint-actual[i])/(delta*H))
            old_life=3*max(0.,ages[i]-paths[i].end)/AGE_MAX
            new_life=3*max(0.,ages[i]-perturbed.end)/AGE_MAX
            add(np.array([2*n+i]),i*dim+control,np.array([(new_life-old_life)/delta]))
            if len(j):
                changed=perturbed.state(pair_times,strict=False)[:,:2]
                derivative=(changed-positions[i,j,k])/delta
                add(pair_rows,i*dim+control,-4*np.sum(unit[i,j,k]*derivative,axis=1)/H)
        span=AGE_MAX-.03 if fixed_stations else AGE_MAX
        col=i*dim+dim-1
        endpoint_velocity=paths[i].state([ages[i]],strict=False)[0,2:] if ages[i]<=paths[i].end+1e-9 else np.zeros(2)
        add(np.array([2*i,2*i+1]),col,span*endpoint_velocity/H)
        if ages[i]>paths[i].end+1e-9:add(np.array([2*n+i]),col,np.array([3*span/AGE_MAX]))
        if len(j):
            young=ages[i]<ages[j]
            vi=velocities[i,j,k];vj=velocities[j,i,k]
            deriv=np.where(young[:,None],(1-f[k,None])*vi+f[k,None]*vj,vi)*span
            add(pair_rows,col,-4*np.sum(unit[i,j,k]*deriv,axis=1)/H)
    return coo_matrix((vv,(ri,ci)),shape=(3*n+n*(n-1)//2*S,n*dim)).tocsr()


# Duck / swan formation inspired by Fig. 5 of Sci. Adv. 11, eadx2826.
# APPEND ONLY: existing PINN, force law, particle diameter, bounds and tolerances are reused.
# Case 1: six fixed wall stations, with individually timed repeated emissions.
# Paper silhouette is a target only; every reported output position is integrated from the PINN.
def _duck_experiment(fixed_stations):
    import hashlib
    from scipy.optimize import linear_sum_assignment
    needed = ('model','VALIDATION','FlowLookup','trace_particle','direct_field',
              'IndependentLauncherPlanner','minimum_separation')
    missing = [k for k in needed if k not in globals()]
    if missing:
        raise RuntimeError('Run the preceding notebook first. Missing: '+', '.join(missing))
    if not VALIDATION.get('passed', False):
        raise RuntimeError('The existing PINN must pass its unchanged flow validation.')
    label = 'dense_duck_six_fixed' if fixed_stations else 'dense_duck_independent_wall_ports'
    folder = Path(RUN)/label
    folder.mkdir(parents=True, exist_ok=True)
    def weight_hash():
        return hashlib.sha256(b''.join(v.detach().cpu().contiguous().numpy().tobytes()
                              for v in model.state_dict().values())).hexdigest()
    before = weight_hash()
    model.eval()
    # Geometry sampled from the largest connected green silhouette in paper Fig. 5F.
    offsets = np.array([[-0.26, 0.015], [-0.24293374, -0.00593682], [-0.23312213, -0.03139331], [-0.22231646, -0.05642416], [-0.20555686, -0.07783393], [-0.18322947, -0.09340163], [-0.1581064, -0.10401281], [-0.13175064, -0.11106757], [-0.10478624, -0.11526665], [-0.07755192, -0.11704108], [-0.05026204, -0.11662201], [-0.02308604, -0.11409617], [0.00380329, -0.10942373], [0.03018374, -0.10243625], [0.05571483, -0.09280968], [0.07980989, -0.08002664], [0.10157517, -0.06359768], [0.11965727, -0.04323063], [0.13151043, -0.01875745], [0.13525348, 0.00817745], [0.1664191, 0.02906821], [0.20820846, 0.05176729], [0.18393911, 0.06426335], [0.16184533, 0.10341195], [0.14832375, 0.1268073], [0.12515881, 0.1407399], [0.09819931, 0.14400884], [0.07184558, 0.13766195], [0.05133105, 0.12011053], [0.04192245, 0.09476641], [0.04434398, 0.06776227], [0.04066485, 0.04145331], [0.02029128, 0.0239246], [-0.02898283, 0.03819258], [-0.05484294, 0.04687081], [-0.0816316, 0.052033], [-0.10885335, 0.0538688], [-0.13609972, 0.05244573], [-0.16295494, 0.04764763], [-0.18884017, 0.03906416], [-0.2128835, 0.027746], [-0.23892754, 0.03523569], [0.123, 0.104], [-0.06755, -0.03407252], [-0.14945, -0.03407252], [0.04165, -0.03407252], [-0.01295, -0.03407252], [0.09625, 0.06049745], [-0.1085, -0.05771501], [-0.1085, -0.01043003], [0.0826, -0.01043003], [-0.1904, -0.01043003], [-0.06755, -0.08135751], [-0.14945, 0.01321247], [-0.06755, 0.01321247], [0.15085, 0.06049745], [-0.01295, -0.08135751], [-0.01295, 0.01321247], [0.0826, 0.03685496], [0.1372, 0.03685496], [-0.12215, -0.08135751], [-0.09485, -0.08135751], [-0.04025, -0.08135751], [-0.1631, -0.05771501]], dtype=float)
    outline = np.array([[-0.26, 0.015], [-0.25669576, 0.01241984], [-0.25371648, 0.00965408], [-0.25103189, 0.00671448], [-0.24861172, 0.00361282], [-0.24642567, 0.00036085], [-0.24444348, -0.00302966], [-0.24263485, -0.00654696], [-0.24096952, -0.01017927], [-0.23941721, -0.01391484], [-0.23794762, -0.0177419], [-0.2365305, -0.02164868], [-0.23513555, -0.02562344], [-0.23373249, -0.02965439], [-0.23229105, -0.03372978], [-0.23078095, -0.03783785], [-0.22917191, -0.04196683], [-0.22743365, -0.04610496], [-0.22553589, -0.05024047], [-0.22344835, -0.05436161], [-0.22114075, -0.05845661], [-0.21858281, -0.0625137], [-0.21574426, -0.06652113], [-0.21259481, -0.07046713], [-0.20910418, -0.07433993], [-0.2052421, -0.07812778], [-0.20097829, -0.08181891], [-0.19628247, -0.08540155], [-0.19112435, -0.08886395], [-0.18547366, -0.09219434], [-0.17930011, -0.09538096], [-0.17208095, -0.0986889], [-0.16436538, -0.1017902], [-0.15632266, -0.1046036], [-0.14797937, -0.10712998], [-0.13936209, -0.10937021], [-0.13049739, -0.11132516], [-0.12141185, -0.11299571], [-0.11213204, -0.11438272], [-0.10268454, -0.11548706], [-0.09309592, -0.11630961], [-0.08339277, -0.11685124], [-0.07360166, -0.11711282], [-0.06374915, -0.11709522], [-0.05386184, -0.11679932], [-0.0439663, -0.11622597], [-0.03408909, -0.11537606], [-0.02425681, -0.11425046], [-0.01449601, -0.11285004], [-0.00483329, -0.11117566], [0.00470479, -0.1092282], [0.01409164, -0.10700854], [0.0233007, -0.10451754], [0.03230538, -0.10175607], [0.04107911, -0.09872501], [0.04959532, -0.09542523], [0.05782742, -0.09185759], [0.06574885, -0.08802298], [0.07333302, -0.08392225], [0.08055336, -0.07955629], [0.08738329, -0.07492596], [0.09223533, -0.07132402], [0.09582981, -0.06849914], [0.09926123, -0.0656348], [0.10253044, -0.06273098], [0.10563832, -0.05978768], [0.10858574, -0.05680492], [0.11137358, -0.05378268], [0.1140027, -0.05072098], [0.11647398, -0.0476198], [0.11878828, -0.04447915], [0.12094648, -0.04129902], [0.12294946, -0.03807943], [0.12479807, -0.03482036], [0.12649319, -0.03152182], [0.1280357, -0.02818381], [0.12942647, -0.02480632], [0.13066636, -0.02138937], [0.13175625, -0.01793294], [0.13269701, -0.01443704], [0.13348951, -0.01090167], [0.13413462, -0.00732683], [0.13463321, -0.00371251], [0.13498616, -5.873e-05], [0.13519433, 0.00363453], [0.13525861, 0.00736726], [0.13517985, 0.01113947], [0.13495893, 0.01495114], [0.13459672, 0.01880229], [0.13409409, 0.02269291], [0.13345192, 0.026623], [0.13434367, 0.02877103], [0.13764011, 0.02828012], [0.14085024, 0.02790305], [0.14397821, 0.02763655], [0.14702815, 0.02747736], [0.1500042, 0.02742221], [0.15291049, 0.02746783], [0.15575118, 0.02761095], [0.15853039, 0.02784831], [0.16125226, 0.02817664], [0.16392094, 0.02859266], [0.16654057, 0.02909313], [0.16911527, 0.02967475], [0.1716492, 0.03033428], [0.17414648, 0.03106843], [0.17661127, 0.03187395], [0.17904769, 0.03274757], [0.18145989, 0.03368601], [0.18385201, 0.03468602], [0.18622818, 0.03574431], [0.18859254, 0.03685764], [0.19094923, 0.03802272], [0.1933024, 0.03923629], [0.19565618, 0.04049508], [0.1980147, 0.04179583], [0.20038211, 0.04313527], [0.20276255, 0.04451012], [0.20516016, 0.04591713], [0.20757907, 0.04735303], [0.21002342, 0.04881454], [0.21156449, 0.05019854], [0.20943319, 0.05118688], [0.20737614, 0.0521668], [0.2053894, 0.0531368], [0.20346906, 0.05409533], [0.20161119, 0.05504088], [0.19981188, 0.05597193], [0.19806721, 0.05688694], [0.19637324, 0.05778439], [0.19472607, 0.05866277], [0.19312176, 0.05952053], [0.19155641, 0.06035616], [0.19002608, 0.06116814], [0.18852686, 0.06195493], [0.18705483, 0.06271502], [0.18560606, 0.06344687], [0.18417663, 0.06414897], [0.18276263, 0.06481979], [0.18136013, 0.0654578], [0.17996521, 0.06606148], [0.17857395, 0.0666293], [0.17718243, 0.06715974], [0.17578673, 0.06765128], [0.17438292, 0.06810239], [0.17296709, 0.06851154], [0.17153531, 0.06887721], [0.17008366, 0.06919788], [0.16860823, 0.06947202], [0.16710509, 0.0696981], [0.16557032, 0.0698746], [0.164, 0.07], [0.1644753, 0.0748878], [0.16471307, 0.07961684], [0.1647192, 0.08418625], [0.16449956, 0.08859515], [0.16406004, 0.09284269], [0.16340651, 0.09692798], [0.16254486, 0.10085015], [0.16148097, 0.10460833], [0.16022073, 0.10820166], [0.15877001, 0.11162926], [0.15713469, 0.11489026], [0.15532067, 0.11798378], [0.15333381, 0.12090896], [0.15118, 0.12366492], [0.14886512, 0.1262508], [0.14639506, 0.12866572], [0.14377569, 0.13090881], [0.14101289, 0.1329792], [0.13811256, 0.13487602], [0.13508056, 0.13659839], [0.13192279, 0.13814545], [0.12864512, 0.13951632], [0.12525343, 0.14071014], [0.12175361, 0.14172603], [0.11815154, 0.14256312], [0.1144531, 0.14322053], [0.11066416, 0.1436974], [0.10679062, 0.14399286], [0.10283836, 0.14410604], [0.09881325, 0.14403605], [0.09455395, 0.14376805], [0.0903895, 0.143277], [0.08638529, 0.14256981], [0.08254263, 0.14165432], [0.07886283, 0.14053839], [0.07534718, 0.13922984], [0.07199701, 0.13773652], [0.06881361, 0.13606628], [0.06579829, 0.13422695], [0.06295236, 0.13222638], [0.06027713, 0.13007241], [0.0577739, 0.12777288], [0.05544399, 0.12533564], [0.05328869, 0.12276852], [0.05130931, 0.12007937], [0.04950717, 0.11727603], [0.04788356, 0.11436634], [0.04643981, 0.11135815], [0.0451772, 0.10825929], [0.04409705, 0.10507761], [0.04320067, 0.10182095], [0.04248937, 0.09849716], [0.04196444, 0.09511406], [0.04162721, 0.09167952], [0.04147897, 0.08820137], [0.04152103, 0.08468744], [0.04175469, 0.08114559], [0.04218128, 0.07758366], [0.04280209, 0.07400948], [0.04361842, 0.07043091], [0.04443802, 0.06739794], [0.04499885, 0.06476122], [0.04535478, 0.06216884], [0.04551343, 0.05962386], [0.04548244, 0.05712934], [0.04526943, 0.05468832], [0.04488201, 0.05230384], [0.04432782, 0.04997897], [0.04361448, 0.04771675], [0.04274961, 0.04552022], [0.04174084, 0.04339245], [0.04059579, 0.04133647], [0.03932209, 0.03935534], [0.03792735, 0.03745211], [0.03641921, 0.03562983], [0.03480529, 0.03389155], [0.03309322, 0.03224031], [0.03129061, 0.03067917], [0.02940509, 0.02921118], [0.02744428, 0.02783939], [0.02541582, 0.02656684], [0.02332732, 0.02539658], [0.02118641, 0.02433168], [0.01900071, 0.02337516], [0.01677785, 0.0225301], [0.01452545, 0.02179952], [0.01225114, 0.0211865], [0.00996254, 0.02069406], [0.00766727, 0.02032527], [0.00537295, 0.02008318], [0.00135979, 0.02183971], [-0.00538954, 0.02619722], [-0.01233662, 0.03021368], [-0.01946294, 0.03389454], [-0.02674997, 0.03724525], [-0.03417921, 0.04027125], [-0.04173214, 0.04297799], [-0.04939023, 0.04537092], [-0.05713499, 0.04745547], [-0.06494788, 0.04923711], [-0.07281039, 0.05072126], [-0.08070402, 0.05191339], [-0.08861023, 0.05281892], [-0.09651052, 0.05344332], [-0.10438637, 0.05379203], [-0.11221925, 0.05387049], [-0.11999067, 0.05368414], [-0.1276821, 0.05323844], [-0.13527502, 0.05253883], [-0.14275092, 0.05159075], [-0.15009128, 0.05039966], [-0.15727759, 0.048971], [-0.16429132, 0.0473102], [-0.17111398, 0.04542273], [-0.17772703, 0.04331403], [-0.18411196, 0.04098953], [-0.19025025, 0.03845469], [-0.1961234, 0.03571496], [-0.20171288, 0.03277577], [-0.20700018, 0.02964258], [-0.21135788, 0.02713972], [-0.21315448, 0.02785511], [-0.21496136, 0.02859085], [-0.21677613, 0.02933647], [-0.21859639, 0.03008154], [-0.22041975, 0.03081558], [-0.2222438, 0.03152814], [-0.22406615, 0.03220876], [-0.2258844, 0.032847], [-0.22769617, 0.03343238], [-0.22949904, 0.03395447], [-0.23129064, 0.03440279], [-0.23306855, 0.0347669], [-0.23483038, 0.03503634], [-0.23657375, 0.03520065], [-0.23829624, 0.03524937], [-0.23999547, 0.03517206], [-0.24166904, 0.03495825], [-0.24331455, 0.03459748], [-0.24492961, 0.03407931], [-0.24651182, 0.03339328], [-0.24805878, 0.03252892], [-0.2495681, 0.03147578], [-0.25103738, 0.03022342], [-0.25246423, 0.02876136], [-0.25384625, 0.02707916], [-0.25518104, 0.02516635], [-0.2564662, 0.02301249], [-0.25769935, 0.02060712], [-0.25887808, 0.01793977]], dtype=float) + TARGET_CENTER
    if fixed_stations:offsets=np.array([[-0.26, 0.015], [-0.23312213, -0.03139331], [-0.20555686, -0.07783393], [-0.1581064, -0.10401281], [-0.10478624, -0.11526665], [-0.07755192, -0.11704108], [-0.05026204, -0.11662201], [0.00380329, -0.10942373], [0.05571483, -0.09280968], [0.10157517, -0.06359768], [0.13151043, -0.01875745], [0.1664191, 0.02906821], [0.20820846, 0.05176729], [0.18393911, 0.06426335], [0.14832375, 0.1268073], [0.09819931, 0.14400884], [0.05133105, 0.12011053], [0.04434398, 0.06776227], [0.02029128, 0.0239246], [-0.05484294, 0.04687081], [-0.10885335, 0.0538688], [-0.16295494, 0.04764763], [-0.2128835, 0.027746], [0.123, 0.104], [-0.06755, -0.03407252], [-0.14791326, -0.02780017], [-0.00215407, -0.04416299], [0.07466556, -0.00307005], [0.04493301, -0.04366703], [0.11206836, 0.02426405], [-0.0690481, -0.07746131], [0.14383834, 0.06833099], [-0.03405189, -0.06264232], [-0.08389559, 0.02662117], [-0.13474764, 0.0266257], [0.08973176, -0.03018215], [-0.17569995, -0.07773162], [0.07310345, 0.135448], [-0.1839997, -0.00343679], [-0.10751375, -0.0027993], [-0.14311241, -8.852e-05], [0.10891594, -0.00363891]],dtype=float)
    targets = offsets + TARGET_CENTER
    n = len(targets)
    gap = DIAMETER+COLLISION_MARGIN
    pair = np.linalg.norm(targets[:,None]-targets[None,:],axis=2)
    np.fill_diagonal(pair,np.inf)
    if pair.min() < gap or np.any(targets <= DIAMETER/2) or np.any(targets >= [L-DIAMETER/2,H-DIAMETER/2]):
        raise ValueError('Target does not fit the original particle size / channel bounds.')
    station_x = np.array([.25,.50,.75])*L
    lookup = FlowLookup(model)
    # This subclass only removes x_launch from the decisions in the fixed-port case.
    # Encoded port 0..5 means B1,B2,B3,T1,T2,T3; positions are never optimized.
    class DuckPlanner(IndependentLauncherPlanner):
        def physical(self,row,port):
            if not fixed_stations:
                return super().physical(row,int(port))
            side=int(port)//3; j=3*side
            c=CONTROL_MID.copy(); c[j]=station_x[int(port)%3]
            q=np.clip(np.asarray(row,float),0,1)
            c[j+1:j+3]=CONTROL_BOUNDS[j+1:j+3,0]+q[:2]*CONTROL_SPAN[j+1:j+3]
            return c,float(.03+q[2]*(AGE_MAX-.03))
        def one_path(self,row,port):
            key=(int(port),*np.round(np.asarray(row),12))
            if key in self.cache:
                self.cache.move_to_end(key); return self.cache[key]
            c,a=self.physical(row,port)
            side=int(port)//3 if fixed_stations else int(port)
            p=trace_particle(self.field,c,side,tmax=max(a,1e-8),
                             rtol=self.rtol,atol=self.atol,max_step=self.max_step)
            if not p.ok: raise RuntimeError('Original particle ODE failed.')
            self.cache[key]=p
            if len(self.cache)>2048:self.cache.popitem(last=False)
            return p
        def endpoints(self,z,ports):
            rows=np.asarray(z).reshape(n,3 if fixed_stations else 4)
            paths=[self.one_path(row,port) for row,port in zip(rows,ports)]
            ages=np.array([self.physical(row,port)[1] for row,port in zip(rows,ports)])
            actual=np.array([p.state([a],strict=False)[0,:2] for p,a in zip(paths,ages)])
            return paths,ages,actual
        def residual(self,z,ports,goal):
            paths,ages,actual=self.endpoints(z,ports)
            out=[*((actual-goal)/H).ravel(),*(3*max(0.,a-p.end)/AGE_MAX for p,a in zip(paths,ages))]
            # Algebraically identical shared-clock samples, batched per trajectory.
            # Avoid repeating SciPy dense-output segment selection for every pair.
            back=np.minimum(ages[:,None],ages[None,:])[:,:,None]*np.linspace(0,1,COLLISION_SAMPLES)
            positions=np.array([p.state((ages[i]-back[i]).ravel(),strict=False)[:,:2].reshape(n,COLLISION_SAMPLES,2)
                                for i,p in enumerate(paths)])
            distances=np.linalg.norm(positions-positions.swapaxes(0,1),axis=-1)
            lower=np.tril_indices(n,-1)
            out.extend((4*np.maximum(0,gap+CLEARANCE_RESERVE-distances[lower])/H).ravel())
            return np.asarray(out)
        def refine_duck(self,z,ports,iterations):
            # Same bounded least-squares / finite-difference method as the existing planner.
            def fun(v):return self.residual(v,ports,targets)
            iteration=[0]
            def jac(v):
                iteration[0]+=1
                if iteration[0]==1 or iteration[0]%5==0:
                    print(label,self.label,'joint iteration',iteration[0],flush=True)
                return sparse_trajectory_jacobian(self,v,ports,targets,fixed_stations)
            return least_squares(fun,np.clip(z,0,1),jac=jac,bounds=(0,1),
                                 max_nfev=iterations,tr_solver='lsmr',ftol=5e-9,xtol=5e-8,gtol=5e-8).x
    coarse=DuckPlanner(lookup,'duck lookup')
    direct=DuckPlanner(direct_field,'duck direct PINN',rtol=ODE_RTOL,atol=ODE_ATOL,max_step=TRAJECTORY_MAX_STEP)
    print(label,':',n,'particles; original diameter =',DIAMETER,flush=True)
    # Reuse only a verified result from the exact same weights, geometry, mode and physics.
    signature=hashlib.sha256((before+json.dumps(CONFIG,sort_keys=True)+str(fixed_stations)+
                  repr(targets.tolist())+repr((AGE_MAX,POSITION_TOLERANCE,REFERENCE_TOLERANCE,gap))).encode()).hexdigest()
    cache_file=folder/'result.json'
    result=None
    if cache_file.exists():
        old=json.loads(cache_file.read_text())
        if old.get('signature')==signature and old.get('success'):
            result=old;print('Reusing verified trajectories from this exact run.',flush=True)
    embedded=_DENSE_PRECOMPUTED.get(label)
    if result is None and embedded and embedded.get('signature')==signature and embedded.get('success'):
        result=embedded
        cache_file.write_text(json.dumps(result,indent=2))
        print('Reusing strict PINN result: original weights, physical settings and dense target signature matched.',flush=True)
    if result is None:
        options=[[] for _ in range(n)]
        if not fixed_stations:
            bank=independent_seed_bank(coarse)
            for i,point in enumerate(targets):
                options[i]=options_for_particle(coarse,bank,point)
        else:
            # Seed trajectories originate at the six exact hardware coordinates.
            banks=[[] for _ in range(6)]
            initializer=DuckPlanner(exact_flow,'analytical initializer')
            for port in range(6):
                side=port//3;j=3*side
                for ux in np.linspace(CONTROL_BOUNDS[j+1,0],CONTROL_BOUNDS[j+1,1],4):
                    for vy in np.linspace(*CONTROL_BOUNDS[j+2],15):
                        row=np.r_[(np.array([ux,vy])-CONTROL_BOUNDS[j+1:j+3,0])/CONTROL_SPAN[j+1:j+3],1.]
                        p=initializer.one_path(row,port)
                        if p.end>.03:
                            aa=np.linspace(.03,p.end,251)
                            banks[port].append((row,p,aa,p.state(aa)[:,:2]))
                print('Seeded fixed port',port+1,flush=True)
            for i,point in enumerate(targets):
                for port in range(6):
                    starts=[]
                    for row,p,aa,xy in banks[port]:
                        dist=np.linalg.norm(xy-point,axis=1);k=int(dist.argmin())
                        q=row.copy();q[2]=(aa[k]-.03)/(AGE_MAX-.03)
                        starts.append((float(dist[k]),q))
                    starts.sort(key=lambda v:v[0])
                    fitted=[]
                    for _,q in starts[:2]:
                        def endpoint_residual(v):
                            p=initializer.one_path(v,port);_,a=initializer.physical(v,port)
                            return np.r_[(p.state([a],strict=False)[0,:2]-point)/H,
                                         3*max(0,a-p.end)/AGE_MAX]
                        fit=least_squares(endpoint_residual,np.clip(q,1e-8,1-1e-8),bounds=(0,1),
                                          max_nfev=24,diff_step=2e-4,ftol=1e-8,xtol=1e-8,gtol=1e-8)
                        row=fit.x;p=initializer.one_path(row,port);_,a=initializer.physical(row,port)
                        if p.end>=a-1e-8:
                            err=float(np.linalg.norm(p.state([a])[0,:2]-point))
                            p=coarse.one_path(row,port)
                            if p.end<a-1e-8:continue
                            err=float(np.linalg.norm(p.state([a],strict=False)[0,:2]-point))
                            fitted.append(dict(row=row,side=port,path=p,age=a,error=err))
                    options[i].extend(sorted(fitted,key=lambda v:v['error'])[:2])
                if not options[i]:raise RuntimeError('No usable fixed-port seed for particle '+str(i+1))
                print('Fixed-port target',i+1,'/',n,'seed error',min(o['error'] for o in options[i]),flush=True)
        def select(start):
            # Same greedy candidate objective and sweeps as choose_independent_seeds;
            # precompute whole option-pair matrices using batched dense output.
            generator=np.random.default_rng(SEED+301+start)
            ids=[int(generator.integers(len(o))) for o in options]
            if start==0:ids=[min(range(len(o)),key=lambda k:o[k]['error']) for o in options]
            pair_cache={}
            def costs_between(i,j):
                if (i,j) not in pair_cache:
                    aa,bb=options[i],options[j]
                    ta=np.array([o['age'] for o in aa]);tb=np.array([o['age'] for o in bb])
                    backwards=np.minimum(ta[:,None],tb[None,:])[:,:,None]*np.linspace(0,1,COLLISION_SAMPLES)
                    pa=np.array([o['path'].state((ta[k]-backwards[k]).ravel())[:,:2].reshape(len(bb),COLLISION_SAMPLES,2)
                                 for k,o in enumerate(aa)])
                    pb=np.array([o['path'].state((tb[k]-backwards[:,k]).ravel())[:,:2].reshape(len(aa),COLLISION_SAMPLES,2)
                                 for k,o in enumerate(bb)]).swapaxes(0,1)
                    distance=np.linalg.norm(pa-pb,axis=-1).min(axis=-1)
                    matrix=20*(np.maximum(0,gap+CLEARANCE_RESERVE-distance)/H)**2
                    pair_cache[i,j]=matrix;pair_cache[j,i]=matrix.T
                return pair_cache[i,j]
            if fixed_stations:
                # Require all six ports BEFORE collision-aware selection, so the remaining
                # particles can adapt. Never impose new assignments after collision search.
                costs=np.array([[min((o['error'] for o in oo if o['side']==port),default=1e6)
                                  for oo in options] for port in range(6)])
                pp,ii=linear_sum_assignment(costs)
                for port,i in zip(pp,ii):
                    ids[i]=min((k for k,o in enumerate(options[i]) if o['side']==port),key=lambda k:options[i][k]['error'])
            for sweep in range(8):
                changed=False
                for i in generator.permutation(n):
                    cost=np.array([(o['error']/H)**2 for o in options[i]])
                    for j in range(n):
                        if i!=j:cost+=costs_between(i,j)[:,ids[j]]
                    if fixed_stations:
                        used=[options[j][ids[j]]['side'] for j in range(n)]
                        old_port=used[i]
                        if used.count(old_port)==1:
                            cost+=np.array([0. if o['side']==old_port else 1e3 for o in options[i]])
                    k=int(cost.argmin())
                    if cost[k]+1e-14<cost[ids[i]]:ids[i]=k;changed=True
                if not changed:break
            chosen=[options[i][k] for i,k in enumerate(ids)]
            z=np.array([o['row'] for o in chosen]).ravel()
            ports=np.array([o['side'] for o in chosen],int)
            dim=3 if fixed_stations else 4
            rows=z.reshape(n,dim)
            return rows.ravel(),ports
        def check(z,ports):
            rows=np.asarray(z).reshape(n,3 if fixed_stations else 4)
            paths=[];refs=[];ages=[];controls=[]
            sides=[int(p)//3 if fixed_stations else int(p) for p in ports]
            for row,port,side in zip(rows,ports,sides):
                c,a=direct.physical(row,port);controls.append(c);ages.append(a)
                paths.append(trace_particle(direct_field,c,side,tmax=a,rtol=2e-9,atol=2e-11,max_step=TRAJECTORY_MAX_STEP/2))
                refs.append(trace_particle(exact_flow,c,side,tmax=a,rtol=2e-10,atol=2e-12,max_step=TRAJECTORY_MAX_STEP/2))
            ages=np.array(ages)
            valid=all(p.ok and p.end>=a-1e-8 for p,a in zip(paths,ages))
            ref_valid=all(p.ok and p.end>=a-1e-8 for p,a in zip(refs,ages))
            actual=np.array([p.state([a],strict=False)[0,:2] for p,a in zip(paths,ages)])
            reference=np.array([p.state([a],strict=False)[0,:2] for p,a in zip(refs,ages)])
            errors=np.linalg.norm(actual-targets,axis=1);ref_errors=np.linalg.norm(reference-targets,axis=1)
            delta=np.linalg.norm(reference-actual,axis=1)
            sep=minimum_separation(paths,ages) if valid else 0.
            ref_sep=minimum_separation(refs,ages) if ref_valid else 0.
            score=max(0,errors.max()-POSITION_TOLERANCE)/H+2*max(0,gap-sep)/H+(0 if valid else 10)
            ok=bool(valid and ref_valid and errors.max()<=POSITION_TOLERANCE and sep>=gap and
                    ref_errors.max()<=REFERENCE_TOLERANCE and delta.max()<=REFERENCE_TOLERANCE and ref_sep>=gap)
            return dict(z=z,ports=ports,paths=paths,refs=refs,ages=ages,controls=controls,sides=sides,
                        actual=actual,reference=reference,errors=errors,ref_errors=ref_errors,
                        delta=delta,sep=sep,ref_sep=ref_sep,score=float(score),success=ok)
        best=None
        for start in range(2):
            z,ports=select(start)
            print(label,'seed selection finished; checking direct PINN',flush=True)
            trial=check(z,ports)
            if not trial['success']:
                z=coarse.refine_duck(z,ports,COARSE_MAX_NFEV)
                trial=check(z,ports)
            if not trial['success']:
                z=direct.refine_duck(z,ports,DIRECT_MAX_NFEV)
                trial=check(z,ports)
            if best is None or (not trial['success'],trial['score'])<(not best['success'],best['score']):best=trial
            print(label,'start',start+1,'max position error',float(trial['errors'].max()),
                  'minimum separation',trial['sep'],'PASS',trial['success'],flush=True)
            if trial['success']:break
        T=float(best['ages'].max()+LEAD_TIME)
        schedules=[];trajectories=[]
        for i,(p,c,side,age,port) in enumerate(zip(best['paths'],best['controls'],best['sides'],best['ages'],best['ports'])):
            j=3*side
            schedules.append(dict(particle=i+1,launcher=['bottom','top'][side],station_id=(('B','T')[side]+str(int(port)%3+1)) if fixed_stations else 'independent',
                x_launch=float(c[j]),y_launch=float(p.state([0])[0,1]),u_launch=float(c[j+1]),v_launch=float(c[j+2]),
                release_time=float(T-age),age_at_target=float(age),target_x=float(targets[i,0]),target_y=float(targets[i,1]),
                actual_x=float(best['actual'][i,0]),actual_y=float(best['actual'][i,1]),position_error=float(best['errors'][i])))
            aa=np.linspace(0,min(age,p.end),501)
            trajectories.append(dict(particle=i+1,side=side,release=float(T-age),flow_controls=c.tolist(),ages=aa.tolist(),states=p.state(aa).tolist()))
        if fixed_stations:
            assert len(set(r['station_id'] for r in schedules))==6
            assert all(r['x_launch'] in station_x for r in schedules)
        result=dict(kind=label,closed=False,success=best['success'],signature=signature,
            reason='all original numerical checks passed' if best['success'] else 'best found; one or more original numerical checks failed',
            targets=targets.tolist(),actual=best['actual'].tolist(),analytical_reference=best['reference'].tolist(),
            target_time=T,max_error=float(best['errors'].max()),rms_error=float(np.sqrt(np.mean(best['errors']**2))),
            max_reference_error=float(best['ref_errors'].max()),max_PINN_reference_delta=float(best['delta'].max()),
            min_pair_distance=float(best['sep']),reference_min_pair_distance=float(best['ref_sep']),required_pair_distance=gap,
            schedules=schedules,paths=trajectories,outline=outline.tolist(),flow_validated=True,
            control_mode='per-particle-independent',
            launch_position_mode='six fixed stations; repeated timed emissions' if fixed_stations else 'independent positions on the original two walls',
            continuous_decisions=(3 if fixed_stations else 4)*n,optimization_replay=[],
            interpretation='One simultaneous particle snapshot; target silhouette is a guide, not simulated fluid.',
            collision_check='Original shared-time sampling plus local distance minimization; no new forces.')
        cache_file.write_text(json.dumps(result,indent=2))
    assert before==weight_hash(),'Neural weights unexpectedly changed.'
    (folder/'weight_integrity.json').write_text(json.dumps(dict(before=before,after=weight_hash(),unchanged=True),indent=2))
    write_schedule(folder/'release_schedule.csv',result['schedules'])
    print(json.dumps({k:result[k] for k in ['kind','success','max_error','min_pair_distance']},indent=2))
    return result

DENSE_FIXED=_duck_experiment(True)
DENSE_INDEPENDENT=_duck_experiment(False)
assert DENSE_FIXED['success'] and DENSE_INDEPENDENT['success']


In [ ]:
from pathlib import Path
import json,numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Circle,Polygon
from matplotlib.path import Path as MplPath
from scipy.interpolate import CubicSpline
ROOT=RUN
GEOM=dict(boundary_count=42,center=np.array([1.83,.14]),outline=np.array(DENSE_FIXED['outline'])-np.array([1.83,.14]))
NB=int(GEOM['boundary_count']);CENTER=GEOM['center'];OUTLINE=GEOM['outline']+CENTER
COLORS=['#087fa4','#087fa4'];INK='#123343';MUTED='#516975'
def load_results():
 rows=[]
 for mode in ['fixed','independent']:
  file=ROOT/mode/'completed_result.json'
  if not file.exists():raise RuntimeError(mode+' computation is not complete')
  r=json.loads(file.read_text())
  if not r['success']:raise RuntimeError(mode+' did not pass the unchanged physical checks')
  rows.append(r)
 return rows

def reconstructed_boundary(r):
 nb=int(r.get('boundary_count',NB));xy=np.array(r['actual'])[:nb];xy=np.vstack([xy,xy[0]])
 ar=np.r_[0,np.cumsum(np.linalg.norm(np.diff(xy,axis=0),axis=1))]
 return CubicSpline(ar,xy,bc_type='periodic')(np.linspace(0,ar[-1],1200))

def geometric_iou(r):
 pred=reconstructed_boundary(r)
 x=np.linspace(CENTER[0]-.29,CENTER[0]+.24,650);y=np.linspace(0,.30,380)
 X,Y=np.meshgrid(x,y);grid=np.c_[X.ravel(),Y.ravel()]
 a=MplPath(OUTLINE).contains_points(grid);b=MplPath(pred).contains_points(grid)
 return float(np.count_nonzero(a&b)/np.count_nonzero(a|b))

def style(ax):
 ax.set(xlim=(CENTER[0]-.285,CENTER[0]+.235),ylim=(-.005,.305),aspect='equal')
 ax.set_axis_off()

def draw_reconstruction(ax,r,centres=True):
 nb=int(r.get('boundary_count',NB));curve=reconstructed_boundary(r);actual=np.array(r['actual']);targets=np.array(r['targets'])
 ax.fill(curve[:,0],curve[:,1],color='#0888ac',lw=0,zorder=2)
 bill=np.flatnonzero(((targets[:nb,0]-CENTER[0])>.15)&((targets[:nb,1]-CENTER[1])<.08))
 if len(bill)>=3:ax.add_patch(Polygon(actual[bill],closed=True,color='#f3ac43',lw=0,zorder=3))
 if centres:ax.scatter(actual[:,0],actual[:,1],s=5,c='#e5f8ff',alpha=.75,lw=0,zorder=4)
 # This eye is the actual position of simulated particle 43, not a target overlay.
 eye=actual[int(r.get('eye_particle',nb))]
 ax.scatter(*eye,s=24,c=INK,edgecolors='#f5fbfe',linewidths=.6,zorder=5)
 style(ax)

def main(results=None,output_directory=None,sources=None):
 results=load_results() if results is None else results
 out=Path(output_directory) if output_directory is not None else ROOT/'outputs'
 out.mkdir(parents=True,exist_ok=True)
 plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':12,'savefig.facecolor':'white'})
 titles=['Six fixed release sites','Independent wall-release positions']
 fig,axes=plt.subplots(1,2,figsize=(12.8,4.6),layout='constrained')
 for i,(ax,r,title) in enumerate(zip(axes,results,titles)):
  draw_reconstruction(ax,r)
  ax.set_title(title+'\n'+str(len(r['actual']))+' simulated particles · geometric IoU '+f'{geometric_iou(r):.1%}',color=INK,pad=12)
  ax.text(.5,.02,f"Max position error {r['max_error']:.2g}  |  Minimum separation {r['min_pair_distance']:.4f}",
          transform=ax.transAxes,ha='center',color=MUTED,fontsize=9)
 fig.suptitle('Duck formation from simulated particle trajectories',fontsize=19,color=INK,weight='bold')
 fig.supxlabel('Filled silhouettes reconstruct simulated boundary landmarks; particles at their physical diameter are shown separately.',fontsize=8,color=MUTED)
 for ext in ['png','pdf','svg']:fig.savefig(out/('duck_presentation.'+ext),dpi=400)
 plt.close(fig)
 fig,axes=plt.subplots(2,3,figsize=(13.5,7.3),layout='constrained')
 for row,(r,title) in enumerate(zip(results,titles)):
  targets=np.array(r['targets']);actual=np.array(r['actual'])
  axes[row,0].fill(OUTLINE[:,0],OUTLINE[:,1],color='#cadbe1',lw=0)
  axes[row,0].scatter(targets[:,0],targets[:,1],s=5,c=INK,lw=0)
  axes[row,0].set_title(title+'\nTarget geometry')
  draw_reconstruction(axes[row,1],r);axes[row,1].set_title('Reconstructed PINN output')
  for xy in actual:axes[row,2].add_patch(Circle(xy,.0125,facecolor='#0888ac',edgecolor='white',lw=.25))
  axes[row,2].set_title('Actual particles · diameter = 0.025')
  for ax in axes[row]:style(ax)
 fig.suptitle('Geometry and physical-particle views',fontsize=18,color=INK,weight='bold')
 fig.supxlabel('Original force law, neural checkpoint, channel dimensions, velocity bounds and collision threshold are unchanged.',fontsize=8,color=MUTED)
 for ext in ['png','pdf']:fig.savefig(out/('duck_validation.'+ext),dpi=320)
 plt.close(fig)
 metrics=[]
 for r,title,mode in zip(results,titles,['fixed','independent']):
  metrics.append(dict(case=title,particles=len(r['actual']),success=r['success'],max_position_error=r['max_error'],
      minimum_separation=r['min_pair_distance'],required_separation=r['required_pair_distance'],geometric_iou=geometric_iou(r),
      max_reference_error=r['max_reference_error'],max_PINN_reference_delta=r['max_PINN_reference_delta']))
  (out/(mode+'_result.json')).write_text(json.dumps(r,indent=2))
  if sources is not None:folder=Path(sources[mode])
  else:
   run=Path((ROOT/mode/'completed_run.txt').read_text());folder=run/r['kind']
  import shutil
  for filename in ['release_schedule.csv','weight_integrity.json','duck_trajectories.png']:
   if filename=='duck_trajectories.png' and not (folder/filename).exists():
    fig,ax=plt.subplots(figsize=(13,3),layout='constrained')
    for trajectory in r['paths']:
     xy=np.array(trajectory['states'])[:,:2];ax.plot(xy[:,0],xy[:,1],lw=.55,alpha=.4,color=['#087fa4','#d46a8a'][trajectory['side']])
    ax.scatter(np.array(r['actual'])[:,0],np.array(r['actual'])[:,1],s=12,color='#087fa4')
    ax.axhline(0,color=INK);ax.axhline(.3,color=INK);ax.set(xlim=(0,2.2),ylim=(-.01,.31),aspect='equal',xlabel='x',ylabel='y',title=title)
    fig.savefig(out/(mode+'_'+filename),dpi=220);plt.close(fig)
   else:shutil.copyfile(folder/filename,out/(mode+'_'+filename))
 (out/'validation_summary.json').write_text(json.dumps(metrics,indent=2))
 print(json.dumps(metrics,indent=2))
 return results


def write_dense_replay(results,filename):
    import json
    from pathlib import Path
    packed=[]
    for r in results:
        q={k:r[k] for k in ['kind','targets','actual','target_time','success','max_error','min_pair_distance']}
        q['boundary_count']=int(r.get('boundary_count',42));q['eye_particle']=int(r.get('eye_particle',q['boundary_count']));q['paths']=[]
        for p in r['paths']:
            ids=list(range(0,len(p['ages']),5))
            if ids[-1]!=len(p['ages'])-1:ids.append(len(p['ages'])-1)
            q['paths'].append(dict(side=p['side'],release=p['release'],ages=[p['ages'][i] for i in ids],xy=[p['states'][i][:2] for i in ids]))
        packed.append(q)
    page=r'''<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Dense duck particle formation</title>
<style>*{box-sizing:border-box}body{margin:0;background:#edf4f7;color:#133445;font:15px system-ui,sans-serif}main{max-width:1180px;margin:24px auto;padding:24px;background:white;border-radius:18px;box-shadow:0 12px 40px #102f4210}h1{margin:0;font-size:29px}p{color:#587080;line-height:1.6}.row{display:flex;gap:14px;align-items:center;flex-wrap:wrap;margin:16px 0}button,select{font:inherit;background:#087fa4;color:white;border:0;border-radius:8px;padding:10px 14px;cursor:pointer}select{background:#eaf3f6;color:#133445}input[type=range]{flex:1;min-width:180px;accent-color:#087fa4}canvas{width:100%;height:470px;border:1px solid #e3edf0;border-radius:12px;display:block}.metric{background:#eef5f8;border-radius:9px;padding:10px 14px;font-size:13px}.muted{font-size:12px;color:#607784}label{cursor:pointer}.footer{border-top:1px solid #e7eef2;padding-top:14px}@media(max-width:650px){main{margin:0;padding:16px;border-radius:0}canvas{height:340px}h1{font-size:23px}}</style>
<main><h1>Duck formation from particle trajectories</h1><p>42 / 64 particles · original frozen neural checkpoint · original force law and particle diameter</p>
<div class="row"><select id="case"><option>Six fixed release sites</option><option>Independent wall-release positions</option></select><span class="metric" id="metrics"></span></div>
<canvas id="scene"></canvas><div class="row"><button id="play">Play releases</button><button id="final">Final shape</button><input id="time" type="range" min="0" max="1" step=".001" value="1"><span id="clock"></span></div>
<div class="row"><label><input id="fill" type="checkbox" checked> Reconstruct final boundary</label><label><input id="full" type="checkbox"> Full channel</label><label><input id="paths" type="checkbox"> Trajectories</label><button id="save">Save image</button></div>
<p class="muted footer">The filled silhouette is a geometric reconstruction of simulated boundary landmarks. With reconstruction off, disks are drawn at their actual physical diameter (0.025). Positions and release times come from the saved ODE trajectories; the shape is a snapshot at the common target time.</p></main>
<script>'use strict';const DATA=__DATA__,$=id=>document.getElementById(id),c=$('scene'),ctx=c.getContext('2d');let fraction=1,playing=false,last=null;const r=()=>DATA[+$('case').value];
function position(p,age){if(age<0||age>p.ages[p.ages.length-1]+1e-7)return null;let lo=0,hi=p.ages.length-1;while(hi-lo>1){let k=(lo+hi)>>1;if(p.ages[k]<=age)lo=k;else hi=k}let f=(age-p.ages[lo])/(p.ages[hi]-p.ages[lo]||1);return p.xy[lo].map((v,j)=>v*(1-f)+p.xy[hi][j]*f)}
function draw(){let R=r(),rect=c.getBoundingClientRect(),w=rect.width,h=rect.height,dpr=Math.min(devicePixelRatio||1,2);c.width=w*dpr;c.height=h*dpr;ctx.setTransform(dpr,0,0,dpr,0,0);ctx.fillStyle='white';ctx.fillRect(0,0,w,h);let full=$('full').checked,b=full?[-.02,2.22,-.015,.315]:[1.545,2.065,-.005,.305],s=Math.min((w-40)/(b[1]-b[0]),(h-40)/(b[3]-b[2])),map=p=>[w/2+(p[0]-(b[0]+b[1])/2)*s,h/2-(p[1]-(b[2]+b[3])/2)*s],time=fraction*R.target_time;
function line(xy,color,width=1){if(!xy.length)return;ctx.beginPath();ctx.strokeStyle=color;ctx.lineWidth=width;xy.forEach((p,i)=>{let q=map(p);i?ctx.lineTo(...q):ctx.moveTo(...q)});ctx.stroke()}
if(full){line([[0,0],[2.2,0]],'#7f99a6',2);line([[0,.3],[2.2,.3]],'#7f99a6',2)}
if($('paths').checked)R.paths.forEach(p=>line(p.xy,p.side?'#d96b8950':'#0786a550',.8));
let positions=R.paths.map(p=>position(p,time-p.release));let filled=$('fill').checked&&fraction>.999;
if(filled){let boundary=R.actual.slice(0,R.boundary_count);ctx.beginPath();boundary.forEach((p,i)=>{let q=map(p);i?ctx.lineTo(...q):ctx.moveTo(...q)});ctx.closePath();ctx.fillStyle='#0888ac';ctx.fill();let bill=boundary.filter((p,i)=>R.targets[i][0]-1.83>.15&&R.targets[i][1]-.14<.08);ctx.beginPath();bill.forEach((p,i)=>{let q=map(p);i?ctx.lineTo(...q):ctx.moveTo(...q)});ctx.closePath();ctx.fillStyle='#f3ac43';ctx.fill()}
positions.forEach((p,i)=>{if(!p)return;let q=map(p);ctx.beginPath();ctx.arc(...q,filled?1.7:.0125*s,0,2*Math.PI);ctx.fillStyle=filled?'#dcf6ff':(R.paths[i].side?'#168cac':'#087fa4');ctx.fill();if(!filled){ctx.strokeStyle='white';ctx.lineWidth=.6;ctx.stroke()}});
if(filled){let eye=map(R.actual[R.eye_particle]);ctx.beginPath();ctx.arc(...eye,3,0,2*Math.PI);ctx.fillStyle='#123343';ctx.fill();ctx.strokeStyle='white';ctx.lineWidth=.6;ctx.stroke()}
$('time').value=fraction;$('clock').textContent=time.toFixed(3)+' / '+R.target_time.toFixed(3);$('metrics').textContent='PASS · max error '+R.max_error.toExponential(2)+' · min separation '+R.min_pair_distance.toFixed(5)}
$('play').onclick=()=>{fraction=0;playing=true;last=null;$('full').checked=true;draw()};$('final').onclick=()=>{fraction=1;playing=false;$('full').checked=false;draw()};$('case').onchange=()=>{fraction=1;playing=false;draw()};$('time').oninput=e=>{fraction=+e.target.value;playing=false;draw()};['fill','full','paths'].forEach(k=>$(k).onchange=draw);$('save').onclick=()=>{let a=document.createElement('a');a.download='duck_particle_formation.png';a.href=c.toDataURL('image/png');a.click()};function frame(ts){if(playing){if(last!==null)fraction=Math.min(1,fraction+(ts-last)/12000);last=ts;if(fraction>=1){playing=false;$('full').checked=false}draw()}requestAnimationFrame(frame)}window.onresize=draw;draw();requestAnimationFrame(frame);
</script></html>'''
    Path(filename).write_text(page.replace('__DATA__',json.dumps(packed,separators=(',',':')).replace('<','\\u003c')))
    return Path(filename)

RESULTS=[DENSE_FIXED,DENSE_INDEPENDENT]
main(RESULTS,RUN/'presentation',dict(fixed=RUN/DENSE_FIXED['kind'],independent=RUN/DENSE_INDEPENDENT['kind']))
replay=write_dense_replay(RESULTS,RUN/'presentation'/'duck_replay.html')
display(Image(filename=str(RUN/'presentation'/'duck_presentation.png')))
display(Image(filename=str(RUN/'presentation'/'duck_validation.png')))
display(FileLink(str(RUN/'presentation'/'duck_presentation.pdf')))
display(FileLink(str(replay)))
archive=shutil.make_archive(str(RUN/'dense_duck_results'),'zip',root_dir=RUN/'presentation')
display(FileLink(archive))
